# Question 1 — prepare annual and monthly panels

One preparation notebook is sufficient: both outputs use the same respondents, cleaning rules, geography lookup and 124-activity schema. The only intentional difference is the official survey weight and time grouping:

- annual panels use `wt_final` and survey wave;
- monthly panels use `wt_time` and interview month.

The outputs retain 372 activity variables (124 activities × three definitions), only three compact non-activity composition features, and Kish `effective_n` as quality metadata. Routed or unavailable activity answers remain missing rather than being changed to zero. No train/test split is stored here; temporal splitting belongs in the forecasting notebook.

In [1]:
from pathlib import Path
import csv
import json

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'physical_activity_analysis' else cwd
OUTPUT_DIR = PROJECT_ROOT / 'physical_activity_analysis' / 'csv_outputs'
WAVES_PATH = PROJECT_ROOT / 'data_integration' / 'waves.json'
LOOKUP_PATH = PROJECT_ROOT / 'Jingyi Hua' / 'data' / 'processed' / 'variable_value_labels_lookup_year7_8.csv'

OUTPUTS = {
    'london_annual': OUTPUT_DIR / 'question1_london_annual.csv',
    'area_annual': OUTPUT_DIR / 'question1_inner_outer_annual.csv',
    'borough_annual': OUTPUT_DIR / 'question1_borough_annual.csv',
    'london_monthly': OUTPUT_DIR / 'question1_london_monthly.csv',
    'area_monthly': OUTPUT_DIR / 'question1_inner_outer_monthly.csv',
    'borough_monthly': OUTPUT_DIR / 'question1_borough_monthly.csv',
}
RAW_MISSINGNESS_DETAIL_OUTPUT = OUTPUT_DIR / 'question1_raw_missingness_by_wave_variable.csv'
RAW_MISSINGNESS_SUMMARY_OUTPUT = OUTPUT_DIR / 'question1_raw_missingness_variable_summary.csv'

assert WAVES_PATH.exists() and LOOKUP_PATH.exists()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPSS_MISSING = list(range(-99, -89))
CITY_OF_LONDON_LA2023 = 59
AREA_LABELS = {1: 'Inner London', 2: 'Outer London'}
AREA_CODES = {1: 'E13000001', 2: 'E13000002'}
FIRST_SURVEY_PERIOD = pd.Period('2015-11', freq='M')
COMPOSITION_FEATURES = ['older_adult_rate', 'limiting_disability_rate', 'online_response_rate']
OUTCOME_COLUMNS = ['inactive_rate', 'fairly_active_rate', 'active_rate']
QUALITY_COLUMNS = ['effective_n']

waves = json.loads(WAVES_PATH.read_text(encoding='utf-8'))
for wave in waves:
    wave['path'] = PROJECT_ROOT / wave['relative_path']
    assert wave['path'].exists(), wave['path']
assert [wave['wave_index'] for wave in waves] == list(range(1, 9))

## 1. Discover the strict common activity schema

Only activity fields present under both source definitions in every one of the eight waves are retained.

In [2]:
def read_header(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        return next(csv.reader(handle))


headers = {wave['wave_index']: read_header(wave['path']) for wave in waves}
months_sets = {
    index: {column.removeprefix('MONTHS_12_') for column in header if column.startswith('MONTHS_12_')}
    for index, header in headers.items()
}
days_sets = {
    index: {column.removeprefix('DAYS10P60GR_') for column in header if column.startswith('DAYS10P60GR_')}
    for index, header in headers.items()
}
common_suffixes = set.intersection(*months_sets.values()) & set.intersection(*days_sets.values())
ACTIVITY_SUFFIXES = [
    column.removeprefix('MONTHS_12_')
    for column in headers[1]
    if column.startswith('MONTHS_12_') and column.removeprefix('MONTHS_12_') in common_suffixes
]
MONTHS_COLUMNS = [f'MONTHS_12_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_COLUMNS = [f'DAYS10P60GR_{suffix}' for suffix in ACTIVITY_SUFFIXES]
MONTHS12_RATE_COLUMNS = [f'MONTHS12_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_ANY_RATE_COLUMNS = [f'DAYS_ANY_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_2PLUS_RATE_COLUMNS = [f'DAYS_2PLUS_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
ACTIVITY_FEATURES = MONTHS12_RATE_COLUMNS + DAYS_ANY_RATE_COLUMNS + DAYS_2PLUS_RATE_COLUMNS

assert len(ACTIVITY_SUFFIXES) == 124
assert len(ACTIVITY_FEATURES) == 372 and len(set(ACTIVITY_FEATURES)) == 372
assert 'HULAHOOP_P27' not in ACTIVITY_SUFFIXES
print(f'Common activities: {len(ACTIVITY_SUFFIXES)}; output activity variables: {len(ACTIVITY_FEATURES)}')

Common activities: 124; output activity variables: 372


## 2. Audit missingness in the raw fields used for Question 1

This audit runs **before** eligibility filtering and weighting, but is deliberately restricted to the raw fields that construct the Question 1 panels:

- `mode`, `LA_2023`, `LondInOut`, `Age9`, `Disab3`, `MEMS7GR_ALL`;
- annual and monthly weights: `wt_final`, `wt_time`;
- the wave-specific month field;
- 124 common `MONTHS_12_*` fields and 124 common `DAYS10P60GR_*` fields used to create the final 372 activity rates.

No unrelated raw questionnaire variables are included. Missingness is separated into:

- blank values parsed by pandas as `NaN`;
- the project's documented SPSS special-missing codes `-99` to `-90`.

The complete Question 1 source-variable table is saved as `question1_raw_missingness_by_wave_variable.csv`, including source fields with zero missing values. A second file provides an across-wave summary. High missingness in routed activity questions can be structural skip-pattern missingness and must not automatically be interpreted as poor data quality or changed to zero.

In [3]:
RAW_AUDIT_CHUNK_SIZE = 5000
Q1_RAW_BASE_COLUMNS = [
    'mode', 'LA_2023', 'LondInOut', 'Age9', 'Disab3',
    'wt_final', 'wt_time', 'MEMS7GR_ALL',
]
SPSS_MISSING_TOKENS = list({
    token
    for code in SPSS_MISSING
    for token in (code, float(code), str(code), f'{code}.0')
})


def audit_raw_wave_missingness(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    required = set(Q1_RAW_BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS)
    q1_columns = [column for column in header if column in required]
    assert len(q1_columns) == 257
    assert set(q1_columns) == required

    blank_counts = np.zeros(len(q1_columns), dtype=np.int64)
    spss_counts = np.zeros(len(q1_columns), dtype=np.int64)
    row_count = 0

    for chunk in pd.read_csv(
        wave['path'], usecols=q1_columns,
        chunksize=RAW_AUDIT_CHUNK_SIZE, low_memory=False
    ):
        chunk = chunk[q1_columns]
        row_count += len(chunk)
        blank_counts += chunk.isna().sum(axis=0).to_numpy(dtype=np.int64)
        spss_counts += chunk.isin(SPSS_MISSING_TOKENS).sum(axis=0).to_numpy(dtype=np.int64)

    total_missing = blank_counts + spss_counts
    assert (total_missing <= row_count).all()
    return pd.DataFrame({
        'wave_index': wave['wave_index'],
        'survey_wave': wave['survey_wave'],
        'source_file': wave['path'].name,
        'variable_position': [header.index(column) + 1 for column in q1_columns],
        'variable': q1_columns,
        'rows_evaluated': row_count,
        'blank_na_count': blank_counts,
        'spss_missing_code_count': spss_counts,
        'total_missing_count': total_missing,
        'observed_count': row_count - total_missing,
        'missing_rate': total_missing / row_count,
        'has_missing': total_missing > 0,
    })


raw_missingness_detail = pd.concat(
    [audit_raw_wave_missingness(wave) for wave in waves], ignore_index=True
).sort_values(['wave_index', 'variable_position']).reset_index(drop=True)

raw_missingness_summary = (
    raw_missingness_detail.groupby('variable', as_index=False)
    .agg(
        waves_present=('wave_index', 'nunique'),
        rows_evaluated=('rows_evaluated', 'sum'),
        blank_na_count=('blank_na_count', 'sum'),
        spss_missing_code_count=('spss_missing_code_count', 'sum'),
        total_missing_count=('total_missing_count', 'sum'),
        observed_count=('observed_count', 'sum'),
    )
)
raw_missingness_summary['waves_absent'] = len(waves) - raw_missingness_summary['waves_present']
raw_missingness_summary['missing_rate'] = (
    raw_missingness_summary['total_missing_count'] / raw_missingness_summary['rows_evaluated']
)
raw_missingness_summary['has_missing'] = raw_missingness_summary['total_missing_count'].gt(0)
raw_missingness_summary = raw_missingness_summary.sort_values(
    ['missing_rate', 'total_missing_count', 'variable'], ascending=[False, False, True]
).reset_index(drop=True)

raw_missingness_detail.to_csv(
    RAW_MISSINGNESS_DETAIL_OUTPUT, index=False, encoding='utf-8-sig'
)
raw_missingness_summary.to_csv(
    RAW_MISSINGNESS_SUMMARY_OUTPUT, index=False, encoding='utf-8-sig'
)

wave_missingness_overview = (
    raw_missingness_detail.groupby(['wave_index', 'survey_wave', 'source_file'], as_index=False)
    .agg(
        rows=('rows_evaluated', 'first'),
        variables=('variable', 'size'),
        variables_with_missing=('has_missing', 'sum'),
        missing_cells=('total_missing_count', 'sum'),
        cells_evaluated=('rows_evaluated', 'sum'),
    )
)
wave_missingness_overview['overall_missing_rate'] = (
    wave_missingness_overview['missing_cells'] / wave_missingness_overview['cells_evaluated']
)

display(wave_missingness_overview.style.format({'overall_missing_rate': '{:.2%}'}))
display(
    raw_missingness_summary.loc[raw_missingness_summary['has_missing']].head(40)
    .style.format({'missing_rate': '{:.2%}'})
)
assert raw_missingness_detail.groupby('wave_index').size().eq(257).all()
assert set(raw_missingness_detail['variable']) <= set(Q1_RAW_BASE_COLUMNS + MONTHS_COLUMNS + DAYS_COLUMNS + ['Month', 'month'])
print(f'Question 1 source-variable audit: {RAW_MISSINGNESS_DETAIL_OUTPUT.name}')
print(f'Across-wave variable summary: {RAW_MISSINGNESS_SUMMARY_OUTPUT.name}')
print(f"Question 1 fields checked: {len(raw_missingness_detail):,} wave-variable rows; "
      f"{raw_missingness_detail['variable'].nunique():,} unique variable names.")

,wave_index,survey_wave,source_file,rows,variables,variables_with_missing,missing_cells,cells_evaluated,overall_missing_rate
0,1,2015/16,active_lives_1516_london_125.csv,19620,257,108,877043,5042340,17.39%
1,2,2016/17,active_lives_1617_london_125.csv,19248,257,109,835796,4946736,16.90%
2,3,2017/18,2017_data_125_activities.csv,15967,257,107,585797,4103519,14.28%
3,4,2018/19,2018_data_125_activities.csv,15889,257,3,1468,4083473,0.04%
4,5,2019/20,1920_london32_stable125.csv,16091,257,3,1349,4135387,0.03%
5,6,2020/21,2021_london32_stable125.csv,16028,257,109,221408,4119196,5.38%
6,7,2021/22,year7_125activities.csv,16139,257,108,411173,4147723,9.91%
7,8,2022/23,year8_125activities.csv,16515,257,108,392122,4244355,9.24%


,variable,waves_present,rows_evaluated,blank_na_count,spss_missing_code_count,total_missing_count,observed_count,waves_absent,missing_rate,has_missing
0,DAYS10P60GR_AIKIDO_S04,8,135497,23121,8260,31381,104116,0,23.16%,True
1,DAYS10P60GR_AIRGUN_S08,8,135497,23121,8260,31381,104116,0,23.16%,True
2,DAYS10P60GR_BOWLSCARPET_U18,8,135497,23121,8260,31381,104116,0,23.16%,True
3,DAYS10P60GR_BOWLSCROWNGREEN_U19,8,135497,23121,8260,31381,104116,0,23.16%,True
4,DAYS10P60GR_BOWLSFLATGREEN_U20,8,135497,23121,8260,31381,104116,0,23.16%,True
5,DAYS10P60GR_BOWLSSHORTMAT_U23,8,135497,23121,8260,31381,104116,0,23.16%,True
6,DAYS10P60GR_CLIMBWALL_R02,8,135497,23121,8260,31381,104116,0,23.16%,True
7,DAYS10P60GR_CRICKETLONG_Q06,8,135497,23121,8260,31381,104116,0,23.16%,True
8,DAYS10P60GR_CRICKETOTH_Q12,8,135497,23121,8260,31381,104116,0,23.16%,True
9,DAYS10P60GR_CRICKETSHORT_Q07,8,135497,23121,8260,31381,104116,0,23.16%,True


Question 1 source-variable audit: question1_raw_missingness_by_wave_variable.csv
Across-wave variable summary: question1_raw_missingness_variable_summary.csv
Question 1 fields checked: 2,056 wave-variable rows; 258 unique variable names.


## 3. Read and clean each wave once

SPSS missing codes become `NaN`. The common eligibility rules are applied first; annual and monthly respondent sets then differ only according to whether their relevant official weight is positive.

In [4]:
BASE_COLUMNS = [
    'mode', 'LA_2023', 'LondInOut', 'Age9', 'Disab3',
    'wt_final', 'wt_time', 'MEMS7GR_ALL',
]


def load_clean_wave(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    usecols = BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS
    assert not (set(usecols) - set(header))

    frame = pd.read_csv(wave['path'], usecols=usecols, low_memory=False)
    frame = frame.rename(columns={month_source: 'month_index'})
    numeric_columns = BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors='coerce')
    frame = frame.replace(SPSS_MISSING, np.nan)

    eligible = (
        frame['MEMS7GR_ALL'].isin([0, 1, 2])
        & frame['LondInOut'].isin([1, 2])
        & frame['LA_2023'].notna()
        & frame['LA_2023'].ne(CITY_OF_LONDON_LA2023)
    )
    frame = frame.loc[eligible, BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS].copy()
    frame.insert(0, 'survey_wave', wave['survey_wave'])
    frame.insert(0, 'year', wave['wave_index'])
    return frame


respondents = pd.concat([load_clean_wave(wave) for wave in waves], ignore_index=True)

lookup = pd.read_csv(LOOKUP_PATH, low_memory=False)
lookup['Code'] = pd.to_numeric(lookup['Code'], errors='coerce')
la_lookup = (
    lookup[lookup['year'].eq(8) & lookup['Variable'].eq('LA_2023')][['Code', 'CodeLabel']]
    .dropna().drop_duplicates('Code')
)
la_lookup['gss_code'] = la_lookup['CodeLabel'].str.split().str[0]
la_lookup['borough'] = la_lookup['CodeLabel'].str.split(n=1).str[1]
respondents['borough'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['borough'])
respondents['gss_code'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['gss_code'])
respondents['inner_outer'] = respondents['LondInOut'].map(AREA_LABELS)
assert respondents[['borough', 'gss_code', 'inner_outer']].notna().all().all()

annual_respondents = respondents.loc[respondents['wt_final'].gt(0)].copy()
monthly_respondents = respondents.loc[respondents['wt_time'].gt(0)].copy()
assert len(annual_respondents) == 135497
assert len(monthly_respondents) == 134916
assert set(monthly_respondents['month_index'].astype(int)) == set(range(1, 97))
assert annual_respondents['borough'].nunique() == monthly_respondents['borough'].nunique() == 32
print(f'Annual respondents: {len(annual_respondents):,}; monthly respondents: {len(monthly_respondents):,}')

Annual respondents: 135,497; monthly respondents: 134,916


## 4. Shared weighted aggregation

The target shares, three composition variables and all 372 activity variables are calculated by the same functions. Passing the weight name explicitly prevents accidental use of the annual weight for monthly estimates or vice versa.

In [5]:
def kish_effective_n(weights):
    weights = np.asarray(weights, dtype=float)
    return float(weights.sum() ** 2 / np.square(weights).sum())


def weighted_binary_share(values, weights, valid_codes, positive_codes):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isin(values, valid_codes) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(np.isin(values[valid], positive_codes), weights=weights[valid]))


def weighted_rate_vector(values, weights, valid_codes, positive_rule):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)[:, None]
    valid = np.isin(values, valid_codes)
    denominator = np.sum(valid * weights, axis=0)
    numerator = np.sum((positive_rule(values) & valid) * weights, axis=0)
    return np.divide(
        numerator, denominator,
        out=np.full(values.shape[1], np.nan, dtype=float),
        where=denominator > 0,
    )


def summarise_group(group, weight_column):
    weights = group[weight_column].to_numpy(float)
    total = weights.sum()
    row = {
        'effective_n': kish_effective_n(weights),
        'inactive_rate': float(weights[group['MEMS7GR_ALL'].eq(0).to_numpy()].sum() / total),
        'fairly_active_rate': float(weights[group['MEMS7GR_ALL'].eq(1).to_numpy()].sum() / total),
        'active_rate': float(weights[group['MEMS7GR_ALL'].eq(2).to_numpy()].sum() / total),
        'older_adult_rate': weighted_binary_share(group['Age9'], weights, range(2, 10), [7, 8, 9]),
        'limiting_disability_rate': weighted_binary_share(group['Disab3'], weights, [1, 2, 3], [1]),
        'online_response_rate': weighted_binary_share(group['mode'], weights, [1, 2], [1]),
    }
    months12_rates = weighted_rate_vector(
        group[MONTHS_COLUMNS], weights, [0, 1], lambda values: values == 1
    )
    days_any_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values >= 1
    )
    days_2plus_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values == 2
    )
    row.update(dict(zip(MONTHS12_RATE_COLUMNS, months12_rates)))
    row.update(dict(zip(DAYS_ANY_RATE_COLUMNS, days_any_rates)))
    row.update(dict(zip(DAYS_2PLUS_RATE_COLUMNS, days_2plus_rates)))
    return row


def aggregate_panel(frame, group_columns, weight_column):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in frame.groupby(grouper, observed=True, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(summarise_group(group, weight_column))
        rows.append(row)
    return pd.DataFrame(rows)

## 5. Annual panels (`wt_final`)

In [6]:
london_annual = aggregate_panel(annual_respondents, ['year', 'survey_wave'], 'wt_final')
london_annual['geography_level'] = 'London'
london_annual['geography_code'] = 'LONDON_32'
london_annual['geography_name'] = 'London excluding City of London'

area_annual = aggregate_panel(
    annual_respondents, ['year', 'survey_wave', 'LondInOut', 'inner_outer'], 'wt_final'
)
area_annual['geography_level'] = 'InnerOuter'
area_annual['geography_code'] = area_annual['LondInOut'].map(AREA_CODES)
area_annual['geography_name'] = area_annual['inner_outer']

borough_annual = aggregate_panel(
    annual_respondents,
    ['year', 'survey_wave', 'gss_code', 'borough', 'inner_outer'],
    'wt_final',
)
borough_annual['geography_level'] = 'Borough'
borough_annual['geography_code'] = borough_annual['gss_code']
borough_annual['geography_name'] = borough_annual['borough']

ANNUAL_ORDER = ['year', 'survey_wave', 'geography_level', 'geography_code', 'geography_name']
MEASURE_ORDER = OUTCOME_COLUMNS + COMPOSITION_FEATURES + QUALITY_COLUMNS + ACTIVITY_FEATURES
london_annual = london_annual[ANNUAL_ORDER + MEASURE_ORDER]
area_annual = area_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_annual = borough_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
display(london_annual)

,year,survey_wave,geography_level,geography_code,geography_name,inactive_rate,fairly_active_rate,active_rate,older_adult_rate,limiting_disability_rate,...,DAYS_2PLUS_RATE_EQUESTOTHER_U17,DAYS_2PLUS_RATE_BOWLSCARPET_U18,DAYS_2PLUS_RATE_BOWLSCROWNGREEN_U19,DAYS_2PLUS_RATE_BOWLSFLATGREEN_U20,DAYS_2PLUS_RATE_BOWLSSHORTMAT_U23,DAYS_2PLUS_RATE_GYMNASTICSONLY_U24,DAYS_2PLUS_RATE_TRAMPOLINING_U25,DAYS_2PLUS_RATE_KARTING_U26,DAYS_2PLUS_RATE_MOTORCYCRACE_U27,DAYS_2PLUS_RATE_MOTORCARRACE_U28
0,1,2015/16,London,LONDON_32,London excluding City of London,0.222108,0.131680,0.646212,0.149331,0.123379,...,0.000532,0.000080,0.000383,0.000943,0.000191,0.005033,0.002317,0.000420,0.000317,0.000000
1,2,2016/17,London,LONDON_32,London excluding City of London,0.229169,0.138296,0.632534,0.147838,0.127761,...,0.001063,0.000209,0.000369,0.000735,0.000449,0.003076,0.003082,0.000545,0.000000,0.000000
2,3,2017/18,London,LONDON_32,London excluding City of London,0.218629,0.113026,0.668344,0.148054,0.130280,...,0.000728,0.000287,0.000000,0.001102,0.000315,0.004981,0.004199,0.000000,0.000262,0.000000
3,4,2018/19,London,LONDON_32,London excluding City of London,0.218696,0.113949,0.667355,0.151436,0.133052,...,0.000391,0.000310,0.000000,0.000410,0.000330,0.003772,0.001591,0.000000,0.000000,0.000000
4,5,2019/20,London,LONDON_32,London excluding City of London,0.241143,0.107682,0.651175,0.150305,0.138444,...,0.000173,0.000258,0.000125,0.000363,0.000024,0.003358,0.001888,0.000000,0.000154,0.000000
5,6,2020/21,London,LONDON_32,London excluding City of London,0.243852,0.106710,0.649438,0.154392,0.145793,...,0.000186,0.000000,0.000026,0.000687,0.000085,0.002831,0.001495,0.000000,0.000000,0.000000
6,7,2021/22,London,LONDON_32,London excluding City of London,0.231198,0.103083,0.665719,0.153714,0.170906,...,0.000326,0.000000,0.000366,0.000928,0.000032,0.004211,0.002250,0.000000,0.000000,0.000079
7,8,2022/23,London,LONDON_32,London excluding City of London,0.237823,0.097904,0.664273,0.155673,0.173844,...,0.000704,0.000117,0.000141,0.001014,0.000169,0.005219,0.002349,0.000096,0.000000,0.000000


## 6. Monthly panels (`wt_time`)

A balanced 96-month grid is retained. The only genuinely empty borough-month is Barking and Dagenham in Month 60; its missing values are not imputed in this preparation step.

In [7]:
wave_labels = {wave['wave_index']: wave['survey_wave'] for wave in waves}
time_reference = pd.DataFrame({'month_index': range(1, 97)})
time_reference['year'] = ((time_reference['month_index'] - 1) // 12 + 1).astype(int)
time_reference['survey_wave'] = time_reference['year'].map(wave_labels)
time_reference['month_of_wave'] = ((time_reference['month_index'] - 1) % 12 + 1).astype(int)
time_reference['period_start'] = [str(FIRST_SURVEY_PERIOD + offset) for offset in range(96)]

borough_reference = (
    monthly_respondents[['LA_2023', 'gss_code', 'borough', 'LondInOut', 'inner_outer']]
    .drop_duplicates().sort_values('borough').reset_index(drop=True)
)
assert len(borough_reference) == 32

london_estimates = aggregate_panel(monthly_respondents, ['month_index'], 'wt_time')
london_monthly = time_reference.merge(london_estimates, on='month_index', how='left', validate='one_to_one')
london_monthly['geography_level'] = 'London'
london_monthly['geography_code'] = 'LONDON_32'
london_monthly['geography_name'] = 'London excluding City of London'

area_reference = pd.DataFrame({
    'LondInOut': [1, 2],
    'inner_outer': [AREA_LABELS[1], AREA_LABELS[2]],
    'geography_code': [AREA_CODES[1], AREA_CODES[2]],
})
area_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LondInOut'], 'wt_time')
area_monthly = time_reference.merge(area_reference, how='cross').merge(
    area_estimates, on=['month_index', 'LondInOut'], how='left', validate='one_to_one'
)
area_monthly['geography_level'] = 'InnerOuter'
area_monthly['geography_name'] = area_monthly['inner_outer']

borough_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LA_2023'], 'wt_time')
borough_monthly = time_reference.merge(borough_reference, how='cross').merge(
    borough_estimates, on=['month_index', 'LA_2023'], how='left', validate='one_to_one'
)
borough_monthly['geography_level'] = 'Borough'
borough_monthly['geography_code'] = borough_monthly['gss_code']
borough_monthly['geography_name'] = borough_monthly['borough']

TIME_ORDER = ['month_index', 'year', 'survey_wave', 'month_of_wave', 'period_start']
GEO_ORDER = ['geography_level', 'geography_code', 'geography_name']
london_monthly = london_monthly[TIME_ORDER + GEO_ORDER + MEASURE_ORDER]
area_monthly = area_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_monthly = borough_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
display(london_monthly.head())

,month_index,year,survey_wave,month_of_wave,period_start,geography_level,geography_code,geography_name,inactive_rate,fairly_active_rate,...,DAYS_2PLUS_RATE_EQUESTOTHER_U17,DAYS_2PLUS_RATE_BOWLSCARPET_U18,DAYS_2PLUS_RATE_BOWLSCROWNGREEN_U19,DAYS_2PLUS_RATE_BOWLSFLATGREEN_U20,DAYS_2PLUS_RATE_BOWLSSHORTMAT_U23,DAYS_2PLUS_RATE_GYMNASTICSONLY_U24,DAYS_2PLUS_RATE_TRAMPOLINING_U25,DAYS_2PLUS_RATE_KARTING_U26,DAYS_2PLUS_RATE_MOTORCYCRACE_U27,DAYS_2PLUS_RATE_MOTORCARRACE_U28
0,1,1,2015/16,1,2015-11,London,LONDON_32,London excluding City of London,0.227642,0.109112,...,0.000000,0.000000,0.0,0.000000,0.000000,0.007924,0.007924,0.000000,0.000000,0.0
1,2,1,2015/16,2,2015-12,London,LONDON_32,London excluding City of London,0.252938,0.159363,...,0.000000,0.000000,0.0,0.002045,0.000000,0.001267,0.000548,0.000000,0.000000,0.0
2,3,1,2015/16,3,2016-01,London,LONDON_32,London excluding City of London,0.256776,0.140119,...,0.001155,0.000332,0.0,0.000435,0.001071,0.002940,0.000480,0.000000,0.000000,0.0
3,4,1,2015/16,4,2016-02,London,LONDON_32,London excluding City of London,0.224726,0.154308,...,0.000342,0.000000,0.0,0.001449,0.000000,0.001044,0.001773,0.004827,0.001852,0.0
4,5,1,2015/16,5,2016-03,London,LONDON_32,London excluding City of London,0.224663,0.112198,...,0.003336,0.000000,0.0,0.000774,0.000963,0.007466,0.000000,0.000000,0.000414,0.0


## 7. Validate and export

These checks cover row uniqueness, valid proportions, target coherence, activity ranges, expected geography coverage and the one known empty monthly cell.

In [8]:
def validate_panel(panel, keys, expected_rows, expected_geographies, expected_months=None, missing_rows=0):
    assert len(panel) == expected_rows
    assert not panel.duplicated(keys).any()
    assert panel['geography_name'].nunique() == expected_geographies
    if expected_months is not None:
        assert set(panel['month_index']) == set(expected_months)
    observed_mask = panel[OUTCOME_COLUMNS].notna().all(axis=1)
    assert int((~observed_mask).sum()) == missing_rows
    observed = panel.loc[observed_mask]
    assert observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES].notna().all().all()
    assert observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES].apply(lambda x: x.between(0, 1)).all().all()
    assert observed['effective_n'].notna().all() and observed['effective_n'].gt(0).all()
    activity_values = observed[ACTIVITY_FEATURES].stack()
    assert activity_values.between(0, 1).all()
    assert observed[ACTIVITY_FEATURES].notna().any().all()
    assert np.allclose(observed[OUTCOME_COLUMNS].sum(axis=1), 1.0, atol=1e-10)


validate_panel(london_annual, ['year'], 8, 1)
validate_panel(area_annual, ['year', 'geography_code'], 16, 2)
validate_panel(borough_annual, ['year', 'geography_code'], 256, 32)
validate_panel(london_monthly, ['month_index'], 96, 1, range(1, 97))
validate_panel(area_monthly, ['month_index', 'geography_code'], 192, 2, range(1, 97))
validate_panel(borough_monthly, ['month_index', 'geography_code'], 3072, 32, range(1, 97), 1)
assert 'City of London' not in set(borough_annual['geography_name'])
assert 'City of London' not in set(borough_monthly['geography_name'])

panels = {
    'london_annual': london_annual,
    'area_annual': area_annual,
    'borough_annual': borough_annual,
    'london_monthly': london_monthly,
    'area_monthly': area_monthly,
    'borough_monthly': borough_monthly,
}
for name, panel in panels.items():
    panel.to_csv(OUTPUTS[name], index=False, encoding='utf-8-sig')

summary = pd.DataFrame([
    {'file': OUTPUTS[name].name, 'rows': len(panel), 'columns': len(panel.columns)}
    for name, panel in panels.items()
])
display(summary)
display(borough_monthly.loc[borough_monthly[OUTCOME_COLUMNS].isna().all(axis=1), TIME_ORDER + GEO_ORDER])
print('Complete: six panels, 372 activity variables, 3 composition variables, and effective_n.')

,file,rows,columns
0,question1_london_annual.csv,8,384
1,question1_inner_outer_annual.csv,16,385
2,question1_borough_annual.csv,256,385
3,question1_london_monthly.csv,96,387
4,question1_inner_outer_monthly.csv,192,388
5,question1_borough_monthly.csv,3072,388


,month_index,year,survey_wave,month_of_wave,period_start,geography_level,geography_code,geography_name
1888,60,5,2019/20,12,2020-10,Borough,E09000002,Barking and Dagenham


Complete: six panels, 372 activity variables, 3 composition variables, and effective_n.


## Modelling hand-off

The borough-month panel is the forecasting base. Predictor values must be lagged within borough before forecasting; `effective_n` is for reliability weighting, not a substantive predictor. The final 12 months are withheld only inside Notebook 02, after feature engineering rules have been fixed.